# The Spark SQL Catalyst Optimizer

## Intermediate concepts and query-plan optimization

Apache Spark 3.5.9 · DataFrames · SQL · Logical and physical planning

# Learning objectives

After this lesson, you should be able to:

- Explain Catalyst's role in Spark SQL and DataFrame processing.
- Identify when parsing, analysis, optimization, and physical planning occur.
- Distinguish logical optimization from physical execution choices.
- Explain important optimization rules and their practical effects.
- Interpret predicate pushdown, partition pruning, column pruning, and data skipping correctly.
- Recognize optimization boundaries and cases where manual design still matters.

# Where Catalyst fits

Catalyst is Spark SQL's framework for representing and transforming query plans.

```text
DataFrame API ─────────────┐
                          ├─> Catalyst planning pipeline ─> executable plan
SQL text ─> SQL parser ───┘
```

It is used for DataFrames, Datasets in JVM languages, SQL queries, catalog tables, temporary views, batch queries, and structured streaming query plans.

# When Catalyst comes into play

Catalyst begins participating while structured expressions are being constructed and prepared for execution.

- DataFrame operations build logical-plan nodes and expressions.
- SQL text is parsed into an unresolved logical plan.
- Analysis resolves tables, columns, functions, and types.
- Optimization rewrites the logical plan.
- Physical planning selects executable operators.
- An action causes the selected plan to execute.

Lazy evaluation gives Catalyst a complete chain of operations to inspect before work begins.

# Catalyst is more than one optimizer step

The name Catalyst is often used only for logical optimization, but the framework supports several planning phases:

1. Parsing
2. Analysis and resolution
3. Logical optimization
4. Physical planning
5. Preparation for execution

Runtime adaptation is handled by Adaptive Query Execution, which works with Spark SQL planning but uses statistics collected after execution has started.

# Plans and expressions are trees

Catalyst represents relational operations as trees.

```text
Aggregate [department], [sum(salary)]
└── Filter [active = true]
    └── Project [department, salary, active]
        └── Relation employees
```

Each node represents an operator. Expressions such as comparisons, arithmetic, function calls, and column references form trees inside those operators.

# Why tree representation matters

A tree provides a consistent structure that rules can examine and transform.

A Catalyst rule generally:

1. Matches one or more plan or expression patterns.
2. Checks whether a safe rewrite applies.
3. Produces an equivalent tree with a different structure.

The result must preserve query semantics. The purpose is to reduce work or produce a form that later planning stages can implement efficiently.

# Rule batches and fixed points

Optimizer rules are organized into batches. Some batches run once; others repeat until the plan stops changing or a configured iteration limit is reached.

Repeated application matters because one rewrite can expose another opportunity.

```text
rule A simplifies an expression
             ↓
rule B can now remove an unnecessary filter
             ↓
rule C can now collapse adjacent projections
```

This is called reaching a fixed point.

# Phase 1: parsing SQL

The SQL parser checks grammar and converts SQL text into an unresolved logical plan.

At this stage, a name such as `ratings.movieId` is syntactically valid but may not yet be connected to a real table or typed column.

DataFrame API expressions normally bypass SQL-text parsing, but they enter the same broader planning pipeline as logical operators and expressions.

# Phase 2: analysis

The analyzer combines the unresolved plan with catalog and schema information.

It resolves:

- Database, table, and view names
- Column references and qualifiers
- Functions and operators
- Aliases and generated attributes
- Wildcards such as `*`
- Data types and legal coercions

The output is an analyzed logical plan whose expressions have defined meaning.

# Analysis failures happen before tasks run

Common analysis errors include:

- Unknown table or column
- Ambiguous column after a join
- Wrong number or type of function arguments
- Unsupported comparison between incompatible types
- Duplicate output names in operations that require uniqueness
- Invalid grouping expressions

These are planning errors, not executor failures. No distributed scan is required to discover them.

# Type coercion

Spark may insert casts when expressions combine compatible but different data types.

Examples include comparisons between integer and long values or arithmetic involving integer and decimal inputs.

Implicit coercion is convenient, but it can:

- Change precision
- Prevent some source-level optimizations
- Produce unexpected nulls for invalid conversions
- Hide weak input schemas

Explicit, correct schemas make analysis and later optimization more predictable.

# Phase 3: logical optimization

Logical optimization rewrites **what** the query means without selecting specific distributed algorithms.

The optimizer may:

- Remove unnecessary work
- Move operations to reduce intermediate data
- Simplify expressions
- Combine compatible operators
- Infer additional safe filters
- Rewrite subqueries or joins

The output remains a logical plan: it says what must be computed, not exactly how executors will compute it.

# Optimization must preserve semantics

Moving or rewriting an operation is valid only when the result remains equivalent.

Semantics that constrain optimization include:

- Null behavior
- Deterministic versus nondeterministic expressions
- Outer-join preservation rules
- Aggregation boundaries
- Error and casting behavior
- Correlated subquery meaning

For example, pushing a filter through an outer join can incorrectly remove rows that the outer join is required to preserve.

# Constant folding

Constant folding evaluates expressions whose inputs are known during planning.

```text
price > (10 * 5)
        ↓
price > 50
```

Other examples include arithmetic constants, literal string operations, and deterministic functions of literals.

This reduces repeated runtime calculation and may expose further simplification opportunities.

# Constant propagation

When a plan establishes that an attribute equals a constant, Catalyst may substitute that information into compatible expressions.

```text
region = 'South' AND upper(region) = 'SOUTH'
```

Knowledge of the first condition can help simplify the second. Constant propagation interacts with boolean simplification, null handling, and predicate inference.

The rewrite is applied only where Spark can preserve expression semantics.

# Boolean simplification

Boolean rules reduce redundant or impossible conditions.

```text
condition AND true   → condition
condition OR false   → condition
NOT(NOT condition)   → condition
```

SQL uses three-valued logic: `TRUE`, `FALSE`, and `UNKNOWN` caused by nulls. Therefore, apparently simple boolean rewrites must respect SQL null semantics.

# Null propagation

Catalyst understands the null behavior of many built-in expressions.

For a null-intolerant expression, a null input guarantees a null result. That knowledge may simplify expressions or remove impossible branches.

Examples include much arithmetic and many comparisons. Other functions deliberately handle nulls, so the same rewrite does not apply universally.

Using built-in expressions exposes null semantics to Catalyst more clearly than opaque external functions.

# Projection pruning

Projection pruning removes columns that are not required by the final result or intermediate operations.

```text
Source: 50 columns
Query: customer_id, order_total
Needed by filters: order_status
Effective requirement: 3 columns
```

Keeping fewer columns reduces memory use, serialization, network traffic, and CPU work. It also enables columnar sources to avoid reading unnecessary data.

# Column pruning versus source column pruning

Two related effects should be distinguished:

- **Plan-level column pruning:** Catalyst stops carrying unused attributes through operators.
- **Source-level column pruning:** a capable reader fetches only required physical columns.

Parquet and ORC can often perform source-level pruning efficiently because they store data by column. CSV is row-oriented and may still need to parse records even when fewer output columns are retained.

# Predicate pushdown inside a logical plan

A predicate is a condition used to filter rows. Catalyst tries to place safe filters as close as possible to the data they constrain.

```text
Project selected columns
└── Join
    ├── Filter active customers
    │   └── Customers
    └── Orders
```

Filtering earlier reduces rows entering later joins, aggregations, and exchanges.

# Predicate pushdown through projections

A filter can often move through a projection when the required source columns remain available.

```text
Filter [price > 100]
└── Project [product_id, price]
    └── Products
```

may become:

```text
Project [product_id, price]
└── Filter [price > 100]
    └── Products
```

This can reduce the number of rows processed by the projection and enable source pushdown.

# Predicate pushdown through joins

A filter that refers only to one join input can often move toward that input.

For an inner join, a customer-country filter can usually be applied to customers before the join. Outer joins require greater care because pushing a predicate may change which unmatched rows survive.

Catalyst considers:

- Join type
- Referenced attributes
- Null-preserving behavior
- Predicate determinism
- Existing join conditions

# Predicate pushdown into a data source

After plan-level movement, Spark asks a capable data source to apply supported predicates while reading.

Benefits may include:

- Returning fewer rows to Spark SQL
- Reading fewer Parquet or ORC row groups
- Sending a `WHERE` clause to a JDBC database
- Reducing decoding and transfer costs

The physical scan plan shows which filters were pushed and which remain for Spark to evaluate.

# Not every predicate can be pushed

Pushdown depends on connector capability and expression form.

Potential barriers include:

- Unsupported functions
- Python or other opaque UDFs
- Complex expressions the source cannot represent
- Casts that change comparison semantics
- Nondeterministic expressions
- Connector limitations

Spark may push part of a compound condition and keep the remaining predicate as a post-scan filter.

# Partition pruning

Partition pruning avoids scanning directory partitions that cannot satisfy a filter.

```text
/sales/year=2025/month=12/
/sales/year=2026/month=01/
/sales/year=2026/month=02/
```

A filter on `year = 2026 AND month = 2` can select only the matching directory when the table or file source recognizes those partition columns.

Partition pruning removes whole input partitions before their data files are read.

# Static and dynamic partition pruning

**Static partition pruning** uses literal conditions known during planning, such as `sale_year = 2026`.

**Dynamic partition pruning** derives filtering values from another side of a join while the query executes. It can avoid scanning fact-table partitions whose keys do not appear in a filtered dimension result.

Dynamic pruning is useful in star-schema queries, but its applicability depends on plan shape, join conditions, statistics, and expected benefit.

# Data skipping

Data skipping avoids reading data blocks or files whose metadata proves that they cannot contain matching rows.

Examples of metadata used for skipping include:

- Minimum and maximum values
- Null counts
- File-level statistics
- Row-group or stripe statistics
- Bloom filters, when supported

Skipping is especially effective when data layout correlates with frequently filtered columns.

# Catalyst and data skipping: separate responsibilities

Catalyst does not personally inspect every Parquet row group and discard it.

The responsibilities are better described as:

```text
Catalyst moves and simplifies predicates
                 ↓
Spark passes supported filters to the source
                 ↓
The reader or table implementation uses metadata
to prune partitions, files, row groups, or pages
```

Catalyst enables the opportunity; the storage format and connector perform the source-specific skipping.

# Partition pruning, pushdown, and skipping compared

| Mechanism | Avoids | Primary information |
| --- | --- | --- |
| Partition pruning | Entire directory or table partitions | Partition values |
| Predicate pushdown | Rows returned from the source | Supported filter expression |
| Column pruning | Unneeded columns | Required attributes |
| Data skipping | Files, row groups, stripes, or pages | Stored statistics or indexes |

One query can benefit from all four mechanisms.

# Combining adjacent projections

Repeated selections or column additions can create several projection nodes. Catalyst may collapse compatible adjacent projections into one.

```text
Project C
└── Project B
    └── Project A
```

can sometimes become one project containing the required final expressions.

This reduces plan complexity and unnecessary intermediate row construction.

# Removing redundant operations

Catalyst may remove operators that cannot affect the result, such as:

- A redundant alias
- An unnecessary projection
- A filter known to be always true
- A no-op repartition in a compatible context
- A sort that is not required by the final semantics

Whether an operation is redundant depends on the surrounding plan. Ordering, partitioning, nullability, and attribute identity can make an apparently unnecessary node meaningful.

# Predicate inference

Join equality can allow Catalyst to infer additional constraints.

```text
orders.customer_id = customers.customer_id
customers.customer_id > 1000
```

The equality may allow a corresponding constraint on `orders.customer_id`.

Inferred predicates can reduce both inputs earlier, but inference must respect join type, null semantics, and expression safety.

# Simplifying casts and expressions

Catalyst can simplify some nested casts and equivalent expressions.

Unnecessary casts add CPU cost and may interfere with pushdown or comparison behavior. However, casts that change precision, range, time-zone interpretation, or null behavior cannot simply be removed.

Consistent schemas across files and tables reduce the number of corrective casts required in the first place.

# Rewriting subqueries

Spark SQL can transform several subquery forms into relational operators that physical planning can implement.

Examples include:

- `EXISTS` as a semi-join pattern
- `NOT EXISTS` as an anti-join pattern
- Scalar subqueries with appropriate aggregation or joins
- Decorrelation of supported correlated predicates

Not every correlated query can be rewritten efficiently. Complex correlation can restrict optimizer choices.

# Phase 4: physical planning

Physical planning maps an optimized logical plan to executable Spark operators.

It decides how to implement:

- Scans
- Joins
- Aggregations
- Sorting
- Limits
- Exchanges and partitioning requirements

Multiple physical candidates may implement the same logical operation. Planning strategies and cost information help select among them.

# Logical rules versus physical strategies

Consider an inner join:

- Logical optimization may push filters into each input and prune unused columns.
- Physical planning may choose broadcast hash join or sort-merge join.
- Execution preparation may add exchanges or sorts required by that choice.
- AQE may revise the join after runtime sizes become available.

These are related decisions, but they occur at different layers.

# Join selection

Physical join selection depends on:

- Equi-join versus non-equality condition
- Join type
- Estimated input sizes
- Broadcast thresholds and hints
- Existing partitioning and ordering
- Availability and quality of statistics

A broadcast join can avoid shuffling the large side, but broadcasting an unexpectedly large relation can exhaust executor memory.

# Cost-Based Optimizer

The Cost-Based Optimizer supplements transformation rules with statistics-based estimates.

Useful statistics include:

- Row count
- Estimated size in bytes
- Distinct counts
- Null counts
- Minimum and maximum values
- Histograms, where available

Cost-based decisions are only as reliable as the available statistics. Missing or stale statistics can produce poor estimates.

# Join reordering

For several inner joins, different join orders can produce the same result but dramatically different intermediate sizes.

```text
(large fact JOIN selective dimension) JOIN other dimension
```

may be cheaper than joining the large inputs first.

Cost-based join reordering uses statistics to search for a lower-cost order. Outer joins and other semantic constraints limit legal reorderings.

# Adaptive Query Execution

Static planning uses estimates available before tasks run. AQE uses observed shuffle statistics to improve the executing query.

AQE can:

- Coalesce small shuffle partitions
- Split or mitigate skewed shuffle partitions
- Convert a planned sort-merge join to a broadcast join
- Optimize local shuffle reads

AQE complements Catalyst planning; it does not eliminate the need for sound schemas, statistics, partitioning, or query design.

# Optimization boundaries

Catalyst can optimize only what it can understand.

Optimization becomes harder around:

- Python and other opaque UDFs
- Arbitrary RDD transformations
- External services with limited connector support
- Nondeterministic expressions
- Side-effecting logic
- Poorly defined or inconsistent schemas

Built-in Spark SQL expressions expose their semantics and are usually easier to optimize.

# Crossing from DataFrame to RDD

Converting a DataFrame to the public RDD API exposes rows to lower-level transformation functions.

Before that boundary, Catalyst can reason about structured columns and expressions. After arbitrary RDD logic begins, Spark SQL generally cannot inspect the meaning of those user functions or push later RDD operations back into the structured source plan.

Use the RDD API when its lower-level control is genuinely needed, not as the default for structured transformations.

# Reading plans with `explain`

An explain output can show several views:

- Parsed logical plan
- Analyzed logical plan
- Optimized logical plan
- Physical plan

Compare them to answer:

- Were filters moved or pushed?
- Were columns pruned?
- Which join strategy was chosen?
- Where are exchanges and sorts?
- Is the plan adaptive?
- Which filters remain after the scan?

# Scan-plan evidence

For file and connector scans, inspect details such as:

- Read schema
- Partition filters
- Pushed filters
- Data filters
- Selected paths or files
- Batch versus row scan

Do not assume that writing a filter early in source code guarantees source pushdown. The physical plan is the evidence of what Spark and the connector accepted.

# Source order is not execution order

Program text may say:

```text
read → add columns → select → filter → action
```

The optimized plan may evaluate the filter near the scan, remove unused derived columns, and combine projections.

DataFrame transformations describe intent. Catalyst determines a semantically equivalent execution-oriented arrangement.

# Catalyst does not solve every performance problem

Catalyst cannot automatically correct all issues, including:

- Severe key skew
- Millions of tiny files
- An unsuitable partitioning strategy
- An unnecessarily large result collected to the driver
- Slow external systems
- Insufficient cluster resources
- Incorrect business logic
- Poor data layout for common access patterns

The optimizer improves a plan within the environment and information it receives.

# Practical optimization habits

- Use explicit and consistent schemas.
- Prefer built-in column and SQL functions.
- Select only required columns.
- Apply selective filters when logically appropriate.
- Use columnar formats for analytical workloads.
- Partition data by useful, reasonably sized dimensions.
- Maintain catalog statistics when cost-based decisions matter.
- Inspect query plans before adding hints.
- Measure with representative data and cluster resources.

# Avoid optimization folklore

Rules such as “always filter first,” “always broadcast,” or “always reduce partitions” are incomplete.

- Catalyst may move a filter regardless of source-code position.
- A broadcast is beneficial only when the broadcast side is suitably small.
- Fewer partitions can reduce overhead but also destroy parallelism.
- Caching helps reused expensive results, not every intermediate DataFrame.

Use plans, metrics, data sizes, and runtime evidence.

# End-to-end example: conceptual flow

```text
SQL joins ratings and movies, filters recent ratings,
groups by movie, and returns the top results
                         ↓
Analyzer resolves tables, columns, functions, and types
                         ↓
Optimizer prunes columns and pushes safe filters
                         ↓
Sources prune partitions and skip irrelevant data
                         ↓
Physical planner selects scans, join, aggregate, and sort
                         ↓
Spark Core executes stages and tasks
                         ↓
AQE may adjust shuffle partitions or join strategy
```

# Key distinctions

| Concept | Primary role |
| --- | --- |
| Analyzer | Resolves meaning and validates expressions |
| Logical optimizer | Rewrites relational intent |
| Physical planner | Chooses executable operators |
| Cost-Based Optimizer | Uses statistics to compare alternatives |
| Adaptive Query Execution | Revises parts of a running query |
| Data-source reader | Performs supported pruning, pushdown, and skipping |
| Spark Core | Schedules stages and tasks across executors |

# Review questions

1. Why does lazy evaluation improve Catalyst's opportunities?
2. What is resolved during analysis?
3. How does projection pruning differ from source column pruning?
4. When is predicate pushdown unsafe?
5. How do partition pruning and data skipping differ?
6. What evidence confirms that a source accepted a pushed filter?
7. Why can a UDF restrict optimization?
8. How do physical planning and AQE differ?
9. Why do statistics matter for join selection and reordering?

# Summary

Catalyst converts structured intent into an analyzed, optimized, and executable query plan.

Its major strengths come from understanding schemas and relational expressions. It can simplify expressions, prune columns, move filters, infer predicates, rewrite plan structures, and support physical strategy selection.

Storage systems contribute partition pruning and data skipping. AQE contributes runtime adaptation. Spark Core contributes distributed execution.

Effective Spark SQL performance comes from these layers working together with sound schemas, data layout, statistics, and query design.